# Reliability measure

Train incidents per million planned km, by line and year. Full years with km data only (2019-2022), plus a January-September comparison that brings in 2023 (km ends 2023-09). After that there is no km, so the last section uses counts up to 2024, by line and by cause. Multi-line and unassigned incidents (1.5% of train rows) can't be put on one line's km, so they are left out.

In [1]:
import pandas as pd
import plotly.express as px

from ds_challenge import plots
from ds_challenge.data import CAUSES, Inc, Km, load_incidents, load_km, train_incidents
from ds_challenge.reliability import per_mkm

inc = load_incidents()
km = load_km()
YEARS = (2019, 2022)
MEASURES = {"incidents_per_mkm": "All incidents", "major_per_mkm": "Major (10+ min)"}

In [2]:
rates = per_mkm(inc[inc[Inc.year].between(*YEARS)], km[km[Km.year].between(*YEARS)])
rates[rates[Inc.line] == "Network"].round(3)

,year,line,incidents,major,mkm,incidents_per_mkm,major_per_mkm,major_share
0,2019,Network,5014,258,89.895,55.776,2.87,0.051
1,2020,Network,3663,247,88.000,41.625,2.807,0.067
2,2021,Network,3515,247,84.302,41.695,2.93,0.07
3,2022,Network,4162,273,84.353,49.340,3.236,0.066


In [3]:
long = rates.rename(columns=MEASURES).melt(
    id_vars=[Inc.year, Inc.line],
    value_vars=list(MEASURES.values()),
    var_name="measure",
    value_name="per_mkm",
)
fig = px.line(
    long,
    x=Inc.year,
    y="per_mkm",
    color=Inc.line,
    facet_col="measure",
    markers=True,
    color_discrete_map={**plots.LINE_COLORS, "Network": "black"},
    labels={Inc.year: "", "per_mkm": "Train incidents per million planned km"},
    title="Per planned km, minor incidents fell in 2020-21 while major ones held",
)
fig.update_yaxes(matches=None, showticklabels=True, rangemode="tozero")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(dtick=1)
fig.update_layout(legend_y=1.08, margin_t=130)  # legend clear of the panel titles
plots.shade_years(fig, plots.COVID_YEARS, "COVID")
plots.save(fig, "reliability_per_mkm", width=1000)

## January-September, 2019-2023

Planned km stops at 2023-09-30, so 2023 is only comparable on the same nine months of every year. 2024 has no km at all and stays out of the per-km view.

In [4]:
jan_sep = per_mkm(
    inc[inc[Inc.year].between(2019, 2023) & (inc[Inc.date].dt.month <= 9)],
    km[km[Km.date].dt.month <= 9],
)
jan_sep.rename(columns=MEASURES).pivot(
    index=Inc.line, columns=Inc.year, values=list(MEASURES.values())
).round(2)

All incidents                                              \
year                  2019       2020       2021       2022       2023   
line                                                                     
Ligne bleue      71.502531  53.759064  43.385384  60.512857  60.033982   
Ligne jaune      84.609016  50.620759  44.265385   64.83982  48.549072   
Ligne orange     47.700223  36.225743  37.647109  38.457722  40.119654   
Ligne verte       57.60578  48.479544  48.249887  57.783469  69.733731   
Network          55.529091   43.14295  42.412445  48.987215   52.90864   

             Major (10+ min)                                          
year                    2019      2020      2021      2022      2023  
line                                                                  
Ligne bleue         5.118749  2.061557  3.416172   4.09101  3.828254  
Ligne jaune         3.808766  2.212929  2.345186  3.212784  2.855828  
Ligne orange        2.133556  2.285341  2.742721  2.178079  2.279145  
Ligne verte         2.776681  3.735511  3.791997  4.354444  7.001379  
Network             2.736823  2.790191  3.167008  3.205513  4.109882

## Counts to 2024: which line, which cause

Major (10+ min) train incidents on a single line, full years. No km exists after 2023-09, so these are counts, not rates. Cause is assigned by a later analysis, which makes it unusable for prediction at declaration but valid for looking back.

In [5]:
major = train_incidents(inc)
major = major[major[Inc.year].between(2019, 2024) & major[Inc.major_10]]
pd.crosstab(major[Inc.line], major[Inc.year], margins=True)

year,2019,2020,2021,2022,2023,2024,All
line,,,,,,,
Ligne bleue,34,20,26,32,27,42,181
Ligne jaune,20,10,9,15,19,28,101
Ligne orange,101,90,101,90,104,153,639
Ligne verte,103,127,111,136,185,142,804
All,258,247,247,273,335,365,1725


In [6]:
green = major[major[Inc.line] == "Ligne verte"]
pd.crosstab(green[Inc.date].dt.quarter.rename("quarter"), green[Inc.year])

year,2019,2020,2021,2022,2023,2024
quarter,,,,,,
1,24,40,36,36,72,39
2,25,22,25,43,38,32
3,18,29,26,21,40,33
4,36,36,24,36,35,38


In [7]:
pd.crosstab(major[Inc.cause_primary].fillna("not recorded"), major[Inc.year], margins=True)

year,2019,2020,2021,2022,2023,2024,All
cause_primary,,,,,,,
Autres,14,14,11,18,9,10,76
Clientèle,126,131,130,172,210,222,991
Exploitation trains,16,4,6,11,10,11,58
Matériel roulant,47,40,35,32,50,46,250
not recorded,1,6,5,4,8,4,28
Équipements fixes,54,52,60,36,48,72,322
All,258,247,247,273,335,365,1725


In [8]:
by_cause = (
    major.assign(cause=major[Inc.cause_primary].map(CAUSES))
    .groupby([Inc.year, "cause"])
    .size()
    .rename("incidents")
    .reset_index()
)
fig = px.line(
    by_cause,
    x=Inc.year,
    y="incidents",
    color="cause",
    markers=True,
    category_orders={"cause": list(CAUSES.values())},
    labels={Inc.year: "", "incidents": "Major (10+ min) train incidents", "cause": ""},
    title="The rise in major incidents is customer-caused; rolling stock is flat",
)
fig.update_yaxes(rangemode="tozero")
fig.update_xaxes(dtick=1)
plots.save(fig, "reliability_major_by_cause")

In [9]:
pd.crosstab(major[Inc.cause_secondary], major[Inc.year]).nlargest(5, 2024)

year,2019,2020,2021,2022,2023,2024
cause_secondary,,,,,,
Méfait volontaire,62,84,80,129,143,152
Blessée ou malade,56,35,35,35,51,55
Service de la voie,25,29,34,18,27,47
MR-73,38,35,23,25,39,28
Service aux trains,22,20,21,13,18,19


In [10]:
# symptom is recorded at declaration: does it tell the same story as the later cause?
pd.crosstab(major[Inc.symptom], major[Inc.year]).nlargest(4, 2024)

year,2019,2020,2021,2022,2023,2024
symptom,,,,,,
Clientèle,99,101,100,125,143,168
Matériel roulant,84,63,54,51,84,100
Équipements fixes,56,65,64,49,81,75
"Feu, fumée, odeur, produit, etc...",10,17,23,38,25,18


## Reading

1. **Volume doesn't explain it.** Planned km fell 6% (2019 to 2022). Until 2022 the major share rose because minor incidents fell (all incidents per km -25% in 2020-21).
2. **From 2023 major incidents rose.** January-September per million km: 2.74 (2019), 4.11 (2023). Counts: 335 (2023), 365 (2024).
3. **The line driving it changes:** green in the first quarter of 2023 (72 against 36 and 39), orange in 2024 (104 to 153).
4. **The rise is customer-caused:** Clientèle 126 to 222, deliberate mischief 62 to 152; rolling stock 47 to 46. The symptom, set at declaration, also shows Clientèle rising (99 to 168).